# Module 06 — Policy Gradients & Actor-Critic

**Reinforcement Learning · Dakar Institute of Technology (DIT)**

So far we learned **value functions** and derived a policy by acting greedily.
**Policy-gradient** methods instead **parameterize the policy directly**,
$\pi_\theta(a\mid s)$, and improve it by gradient *ascent* on expected return. This
naturally handles **stochastic** and **continuous** action spaces (where a `max`
over actions is intractable).

### The policy gradient theorem
$$\nabla_\theta J(\theta) = \mathbb{E}\big[\nabla_\theta \log \pi_\theta(a\mid s)\,\Psi_t\big]$$
where the weight $\Psi_t$ can be:
- the **return** $G_t$ -> **REINFORCE**,
- the **advantage** $A(s,a)=Q(s,a)-V(s)$ -> **Actor-Critic**.

**Actor-Critic** has two networks:
- the **actor** $\pi_\theta$ chooses actions,
- the **critic** $V_\phi$ estimates value and provides a low-variance baseline via
  the TD error $\delta = r + \gamma V(s') - V(s)$, which is an estimate of the advantage.

### Learning objectives
1. Implement **REINFORCE** (Monte Carlo policy gradient) on CartPole.
2. Add a **critic** -> **Actor-Critic (A2C)**; see the variance drop.
3. Reuse the *same* Actor-Critic to solve a much harder task, **LunarLander-v3**.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "rl_helpers").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
from rl_helpers import set_seed, plot_learning_curve, animate_policy

set_seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. REINFORCE on CartPole

The actor is a network outputting action **logits**; we sample from the resulting
categorical distribution. After each episode we compute discounted returns $G_t$
and take a gradient step on the loss
$$L(\theta) = -\sum_t \log\pi_\theta(a_t\mid s_t)\,G_t.$$
(We *minimise* the negative, i.e. ascend the gradient.) Normalising the returns is a
standard variance-reduction trick.

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, n_obs, n_act, hidden=128):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(n_obs, hidden), nn.ReLU(),
                                nn.Linear(hidden, n_act))
    def forward(self, x):
        return self.fc(x) # logits

def select_action(policy, state):
    logits = policy(torch.tensor(state, dtype=torch.float32, device=device))
    dist = Categorical(logits=logits)
    action = dist.sample()
    return action.item(), dist.log_prob(action)

In [ ]:
def reinforce(n_episodes=400, gamma=0.99, lr=1e-3):
    env = gym.make("CartPole-v1")
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n).to(device)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    ep_returns = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=ep)
        log_probs, rewards = [], []
        done = False
        while not done:
            a, logp = select_action(policy, s)
            s, r, term, trunc, _ = env.step(a)
            done = term or trunc
            log_probs.append(logp)
            rewards.append(r)
        # TODO: compute discounted returns G_t for each step (walk backwards)
        # TODO: convert to a tensor and normalise (subtract mean, divide by std)
        # TODO: loss = -(sum of log_prob_t * G_t); backprop and step
        raise NotImplementedError("Implement the REINFORCE update")
        ep_returns.append(sum(rewards))
    env.close()
    return policy, ep_returns

reinforce_policy, reinforce_returns = reinforce(n_episodes=400)
plot_learning_curve(reinforce_returns, window=20, title="REINFORCE on CartPole-v1")
plt.show()

## 2. Actor-Critic (A2C)

REINFORCE is unbiased but **high variance**. Add a **critic** $V_\phi(s)$ and replace
the raw return with the **advantage** $A_t = G_t - V_\phi(s_t)$:
- **Actor loss:** $-\log\pi_\theta(a_t\mid s_t)\,A_t$ (advantage weighting).
- **Critic loss:** regress $V_\phi(s_t)$ toward the return $G_t$.

We write a **general** `train_a2c(env_id, ...)` that works for *any* discrete-action
Gymnasium environment — we'll run it on CartPole now and on LunarLander in Part 3.
One shared trunk feeds two heads (actor + critic); an **entropy bonus** keeps the
policy exploring, and we **clip gradients** for stability on the harder task.

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, n_obs, n_act, hidden=128):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(n_obs, hidden), nn.Tanh(),
                                    nn.Linear(hidden, hidden), nn.Tanh())
        self.actor = nn.Linear(hidden, n_act)
        self.critic = nn.Linear(hidden, 1)
    def forward(self, x):
        h = self.shared(x)
        return self.actor(h), self.critic(h)

def train_a2c(env_id, n_episodes=400, gamma=0.99, lr=3e-3, ent_coef=0.01):
    env = gym.make(env_id)
    net = ActorCritic(env.observation_space.shape[0], env.action_space.n).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    ep_returns = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=ep)
        log_probs, values, rewards, entropies = [], [], [], []
        done = False
        while not done:
            logits, value = net(torch.tensor(s, dtype=torch.float32, device=device))
            dist = Categorical(logits=logits)
            a = dist.sample()
            s, r, term, trunc, _ = env.step(a.item())
            done = term or trunc
            log_probs.append(dist.log_prob(a))
            values.append(value.squeeze())
            rewards.append(r)
            entropies.append(dist.entropy())
        # discounted returns, then standardise them (variance reduction)
        G, returns = 0.0, []
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        values = torch.stack(values)
        # TODO: advantages = returns - values.detach()
        # TODO: actor_loss  = -(sum of log_prob_t * advantage_t)
        # TODO: critic_loss = MSE(values, returns)   (use reduction="sum")
        # TODO: entropy_bonus = sum of the per-step entropies
        # TODO: loss = actor_loss + 0.5*critic_loss - ent_coef*entropy_bonus
        raise NotImplementedError("Implement the actor-critic update")
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
        opt.step()
        ep_returns.append(sum(rewards))
        if (ep + 1) % 50 == 0:
            print(f"ep {ep+1:4d} | avg(last50) return {np.mean(ep_returns[-50:]):.0f}")
    env.close()
    return net, ep_returns

ac_net, ac_returns = train_a2c("CartPole-v1", n_episodes=400)
fig, ax = plt.subplots(figsize=(8, 4))
plot_learning_curve(reinforce_returns, window=20, ax=ax, label="REINFORCE")
plot_learning_curve(ac_returns, window=20, ax=ax, label="Actor-Critic",
                    title="REINFORCE vs Actor-Critic on CartPole")
plt.show()

In [ ]:
# Watch the trained actor-critic balance the pole
render_env = gym.make("CartPole-v1", render_mode="rgb_array")
def ac_policy(s):
    with torch.no_grad():
        logits, _ = ac_net(torch.tensor(s, dtype=torch.float32, device=device))
        return int(logits.argmax())
animate_policy(render_env, ac_policy, max_steps=500, fps=30, path="a2c_cartpole.gif")

## 3. A harder task: LunarLander

**LunarLander-v3** is a genuine control challenge: fire the lander's engines to land
softly between the flags. The observation is 8-dimensional (position, velocity,
angle, angular velocity, and two leg-contact flags) and there are **4 discrete
actions** (do nothing, fire left / main / right engine). Reward combines a landing
bonus, fuel cost, and a crash penalty.

The beautiful part: we change **nothing** in the algorithm — we call the *same*
`train_a2c` on a different `env_id`. Watch the average return climb from about
$-150$ toward $0$ as the agent learns to fire its engines and slow its descent.

> This trains for ~600 episodes and takes several minutes on CPU. It won't fully
> "solve" LunarLander (that needs a bigger budget / stronger methods), but the
> upward learning trend is unmistakable, and the rendered agent visibly learns to
> control its descent. Increase `n_episodes` for a better lander.

In [ ]:
ll_net, ll_returns = train_a2c("LunarLander-v3", n_episodes=600, gamma=0.99, lr=7e-4)
plot_learning_curve(ll_returns, window=30, title="Actor-Critic on LunarLander-v3")
plt.show()
print(f"Best 50-episode average: {max(np.convolve(ll_returns, np.ones(50)/50, 'valid')):.0f}")

In [ ]:
# Watch the trained lander try to touch down between the flags
render_env = gym.make("LunarLander-v3", render_mode="rgb_array")
def ll_policy(s):
    with torch.no_grad():
        logits, _ = ll_net(torch.tensor(s, dtype=torch.float32, device=device))
        return int(logits.argmax())
animate_policy(render_env, ll_policy, max_steps=400, fps=30, path="a2c_lunarlander.gif")

### A note on continuous actions
CartPole and LunarLander both have **discrete** actions, so the actor output a
categorical distribution. Policy gradients extend *directly* to **continuous**
actions (a torque, a steering angle) — the only change is the policy distribution:
the actor outputs the **mean** (and standard deviation) of a **Gaussian**, you
`sample()` a real-valued action, and `log_prob(action)` plugs into the exact same
loss. Value-based methods can't do this easily, because $\max_a Q(s,a)$ over a
continuous range is intractable. This is why the policy-gradient family underlies
modern continuous-control algorithms.

## Recap
- **Policy-gradient** methods optimise a parameterised policy directly.
- **REINFORCE** uses the full return: unbiased but high variance.
- **Actor-Critic** subtracts a learned **baseline** (the critic) and uses the
  **advantage**, cutting variance and learning faster.
- The *same* Actor-Critic code scales from CartPole to the much harder
  **LunarLander** — only the environment id changes.
- Swapping the categorical policy for a **Gaussian** is all it takes to handle
  continuous actions.

### Going further (optional, same environments)
- Add a larger **entropy bonus** or tune `lr` and watch stability change.
- Give LunarLander a bigger `n_episodes` budget and see if you can reach an average
  return above +100.

**That's the whole course** — from tabular gridworlds to deep actor-critic!